In [2]:
import pytest
import requests
import json
from pymeos import *
import urllib.parse
import time
import pandas as pd
import ijson
import os
import folium as fl
import geopandas as gpd
from pyproj import Transformer
from folium import Figure
from IPython.display import display, HTML
from shapely.geometry import Point, LineString
HOST = "http://localhost:8080"

pymeos_initialize()

script_dir = os.getcwd() 
data= os.path.join(script_dir, "data", "trajectories_mf1.json")

In [3]:
def iter_features_from_index(file_path, start_index=0):
    
    with open(file_path, 'rb') as f:
        objects = ijson.items(f, 'item')
        for i, obj in enumerate(objects):
            if i >= start_index:
                yield obj

def json_to_gdf(features,geoms=None,ids=None):
    if geoms is None:
        geoms = []
    if ids is None:
        ids = []


    for feature in features:
        feature_id = feature.get("id")

        coords = []
        if "geometry" in feature and feature["geometry"]:
            for tg in feature["geometry"]:
                coords.extend(tg.get("coordinates", []))

        if coords:
            if len(coords) == 1:
                geom = Point(coords[0])
            else:
                geom = LineString(coords)

            geoms.append(geom)
            ids.append(feature_id)

    gdf = gpd.GeoDataFrame({"id": ids}, geometry=geoms, crs="EPSG:25832")
    gdf = gdf.to_crs(epsg=4326)
    return gdf




from shapely.geometry import shape

def json_to_gdf2(features):
    geoms = []
    ids = []

    for feature in features:
        geom = feature.get("geometry")
        if not geom:
            continue

        try:
            shapely_geom = shape(geom)
        except Exception:
            continue

        geoms.append(shapely_geom)
        ids.append(feature.get("id"))

    gdf = gpd.GeoDataFrame({"id": ids}, geometry=geoms, crs="EPSG:4326")
    # gdf = gpd.GeoDataFrame({"id": ids}, geometry=geoms, crs="EPSG:25832")
    # gdf = gdf.to_crs(epsg=4326)
    return gdf
collection_id = "ships"

##### Delete collections

In [4]:
resp = requests.delete(f"{HOST}/collections/{"ships"}")

print(resp.status_code)
resp = requests.delete(f"{HOST}/collections/{"planes"}")

print(resp.status_code)


204
204


###                   MobilityAPI DEMO

###                   Ships Collection

#### 1.POST Moving features collections 

In [5]:
collections = [
    {
        "title": "ships",
        "updateFrequency": 1000,
        "description": "Collection of ship trajectories (Trips)",
        "itemType": "movingfeature",
    },
    {
        "title": "planes",
        "updateFrequency": 1000,
        "description": "Collection of planes trajectories (Trips)",
        "itemType": "movingfeature",
    }
]


created = []
print("Created collections:")
for col in collections:
    resp = requests.post(f"{HOST}/collections", json=col)
    # log_request_response(f"Create collection {col['title']}", resp)
    
    assert resp.status_code in (201, 409)
    print("RESPONSE CODE:", resp.status_code)
    location = resp.headers.get("Location")
    print("LOCATION:", location)


Created collections:
RESPONSE CODE: 201
LOCATION: http://localhost:8080/collections/ships
RESPONSE CODE: 201
LOCATION: http://localhost:8080/collections/planes


##### 1.GET Moving features collections 

In [6]:

resp = requests.get(f"{HOST}/collections")

# assert resp.status_code == 200
resp_json = resp.json()
collection_ids = [c["id"] for c in resp_json .get("collections", [])]
print("GET COLLECTIONS:")
print("RESPONSE CODE:", resp.status_code)

print("Response object",json.dumps(resp_json, indent=2))

GET COLLECTIONS:
RESPONSE CODE: 200
Response object {
  "collections": [
    {
      "id": "planes",
      "title": "planes",
      "description": "Collection of planes trajectories (Trips)",
      "itemType": "movingfeature",
      "updateFrequency": 1000,
      "extent": null,
      "links": [
        {
          "href": "http://localhost:8080/collections/planes",
          "rel": "self",
          "type": "application/json"
        }
      ]
    },
    {
      "id": "ships",
      "title": "ships",
      "description": "Collection of ship trajectories (Trips)",
      "itemType": "movingfeature",
      "updateFrequency": 1000,
      "extent": null,
      "links": [
        {
          "href": "http://localhost:8080/collections/ships",
          "rel": "self",
          "type": "application/json"
        }
      ]
    },
    {
      "id": "prop_test",
      "title": "prop_test",
      "description": "Collection for property testing",
      "itemType": "movingfeature",
      "updateFre

#### 2.POST Vessels (Moving Features) 

In [7]:

batch_size = 20 # max96
created_count = 0
batch = []

for obj in iter_features_from_index(data , start_index=0):
    batch.append(obj)
    if len(batch) >= batch_size:
        features = []
        for o in batch:
            features.append({
                "type": "Feature",
                "id": str(o["mmsi"]),
                "properties": o["properties"],
                "crs": {
                    "type": "name",
                    "properties": {
                        "name": "urn:ogc:def:crs:EPSG::25832"
                    }
                },
                "trs": {
                    "type": "Link",
                    "properties": {
                        "type": "ogcdef",
                        "href": "http://www.opengis.net/def/uom/ISO-8601/0/Gregorian"
                    }
                },
                "temporalGeometry": json.loads(o["trajectory"]),
                "temporalProperties": [{
                    "datetimes": [
                        "2011-07-14T22:01:01.450Z",
                        "2011-07-14T23:01:01.450Z",
                        "2011-07-15T00:01:01.450Z"
                    ],
                    "length": {
                        "type": "Measure",
                        "form": "http://qudt.org/vocab/quantitykind/Length",
                        "values": [1, 2.4, 1],
                        "interpolation": "Linear",
                        "description": "description1"
                    },
                    "discharge": {
                        "type": "Measure",
                        "form": "MQS",
                        "values": [3, 4, 5],
                        "interpolation": "Step"
                    }
                },
                {
                    "datetimes": [
                        "2011-07-15T23:01:01.450Z",
                        "2011-07-16T00:01:01.450Z"
                    ],
                    "camera": {
                        "type": "Image",
                        "values": [
                        "http://.../example/image1",
                        "VBORw0KGgoAAAANSUhEU......"
                        ],
                        "interpolation": "Discrete"
                    },
                    "labels": {
                        "type": "Text",
                        "values": ["car", "human"],
                        "interpolation": "Discrete"
                    }
                }]
            })
        
        feature_collection = {
            "type": "FeatureCollection",
            "features": features
        }
        
        resp = requests.post(
            f"{HOST}/collections/{collection_id}/items",
            json=feature_collection,
            headers={"Content-Type": "application/json"}
        )
        
        if resp.status_code in (201, 409):
            created_count += len(batch)
            print(f"Batch: Created {len(batch)} features")
        batch = [] 

feature_collection = {
        "type": "FeatureCollection",
        "features": features
    }
    
resp = requests.post(
    f"{HOST}/collections/{collection_id}/items",
    json=feature_collection,
    headers={"Content-Type": "application/json"}
)
print("POST MOVING FEATURES:")
print("RESPONSE CODE:", resp.status_code)
resp_json = resp.json()
# print(resp.headers)
# print("Response object",json.dumps(resp_json, indent=2))
        


Batch: Created 20 features
Batch: Created 20 features
Batch: Created 20 features
Batch: Created 20 features
Batch: Created 20 features
Batch: Created 20 features
Batch: Created 20 features
POST MOVING FEATURES:
RESPONSE CODE: 201


In [10]:


print(json.dumps({
    "type": "FeatureCollection",
    "features": [features[0]]
}, indent=2))

{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "id": "305393000",
      "properties": {
        "IMO": "9433353",
        "Name": "ALEXANDER",
        "ShipType": "Cargo",
        "Destination": "SEHAN"
      },
      "crs": {
        "type": "name",
        "properties": {
          "name": "urn:ogc:def:crs:EPSG::25832"
        }
      },
      "trs": {
        "type": "Link",
        "properties": {
          "type": "ogcdef",
          "href": "http://www.opengis.net/def/uom/ISO-8601/0/Gregorian"
        }
      },
      "temporalGeometry": {
        "type": "MovingPoint",
        "crs": {
          "type": "Name",
          "properties": {
            "name": "EPSG:25832"
          }
        },
        "coordinates": [
          [
            592809.570414379,
            6050356.972208065
          ],
          [
            592809.570414379,
            6050356.972208065
          ],
          [
            592816.1406108743,
            605

#### GET BY COLLECTION ID (single collection)

In [123]:
collection_id = "ships"
resp = requests.get(f"{HOST}/collections/{collection_id}")

assert resp.status_code == 200

print("GET MOVING FEATURES:")
print("RESPONSE CODE:", resp.status_code)
resp_json = resp.json()

# print(resp.headers)
print("Response object",json.dumps(resp_json, indent=2))



GET MOVING FEATURES:
RESPONSE CODE: 200
Response object {
  "id": "ships",
  "title": "ships",
  "description": "Collection of ship trajectories (Trips)",
  "itemType": "movingfeature",
  "updateFrequency": 1000,
  "extent": {
    "spatial": {
      "bbox": [
        25832.0,
        406269.144991072,
        5943153.046693679,
        1125758.0693013691
      ],
      "crs": {
        "type": "name",
        "properties": {
          "name": "urn:ogc:def:crs:EPSG::25832"
        }
      }
    },
    "temporal": {
      "interval": [
        "2024-08-07 02:00:00+02",
        "2024-08-08 01:59:58+02"
      ],
      "trs": {
        "type": "Link",
        "properties": {
          "href": "http://www.opengis.net/def/uom/ISO-8601/0/Gregorian",
          "type": "ogcdef"
        }
      }
    }
  },
  "links": [
    {
      "href": "http://localhost:8080/collections/ships",
      "rel": "self",
      "type": "application/json"
    }
  ]
}


#### REPLACE by COLLECTION ID (single collection)

In [124]:
#no if match handling as per req 10
update_data = {
    "id":"blabla", #req 10 C ignore id
    "title": "Vessels",
    "description": "Updated collection for vessels ...",
    "updateFrequency":5000 #req 7 C ignore updatefrequency
}

resp = requests.put(f"{HOST}/collections/{collection_id}", json=update_data)

assert resp.status_code in (200,204)
print("GET MOVING FEATURES:")
print("RESPONSE CODE:", resp.status_code)


GET MOVING FEATURES:
RESPONSE CODE: 204


In [125]:
resp = requests.get(f"{HOST}/collections/{collection_id}")

assert resp.status_code == 200

print("GET MOVING FEATURES:")
print("RESPONSE CODE:", resp.status_code)
resp_json = resp.json()

# print(resp.headers)
print("Response object",json.dumps(resp_json, indent=2))

GET MOVING FEATURES:
RESPONSE CODE: 200
Response object {
  "id": "ships",
  "title": "Vessels",
  "description": "Updated collection for vessels ...",
  "itemType": "movingfeature",
  "updateFrequency": 1000,
  "extent": {
    "spatial": {
      "bbox": [
        25832.0,
        406269.144991072,
        5943153.046693679,
        1125758.0693013691
      ],
      "crs": {
        "type": "name",
        "properties": {
          "name": "urn:ogc:def:crs:EPSG::25832"
        }
      }
    },
    "temporal": {
      "interval": [
        "2024-08-07 02:00:00+02",
        "2024-08-08 01:59:58+02"
      ],
      "trs": {
        "type": "Link",
        "properties": {
          "href": "http://www.opengis.net/def/uom/ISO-8601/0/Gregorian",
          "type": "ogcdef"
        }
      }
    }
  },
  "links": [
    {
      "href": "http://localhost:8080/collections/ships",
      "rel": "self",
      "type": "application/json"
    }
  ]
}


##### 2.GET Vessels (Moving Features) 

In [1]:
collection_id = "ships"
resp = requests.get(f"{HOST}/collections/{collection_id}/items")

assert resp.status_code == 200

print("GET MOVING FEATURES:")
print("RESPONSE CODE:", resp.status_code)
resp_json = resp.json()
print( len(resp_json["features"]))
# print(resp.headers)
print("Response object",json.dumps(resp_json, indent=2))



NameError: name 'requests' is not defined

In [ ]:
resp = requests.get(f"{HOST}/collections/{collection_id}/items?limit=144")
assert resp.status_code == 200
data = resp.json()
features = data.get("features", [])
import geopandas as gpd
from shapely.geometry import LineString, Point
geoms = []
ids = []




In [ ]:
gdf = json_to_gdf(features,geoms, ids)

fig = Figure(width="900px", height="400px")
m = fl.Map(location=[54.6, 11.0], zoom_start=6, tiles="cartodbpositron",width="100%", height="500px")
fig.add_child(m)
fl.GeoJson(
    gdf,
    name="All ships",  
    tooltip=fl.GeoJsonTooltip(fields=["id"], aliases=["Ship ID:"]),
    style_function=lambda x: {"color": "brown", "weight": 1,  "fillOpacity": 0.6}
).add_to(m)


m

#### 3.Get ships in a port by bbox  

In [ ]:
PORT_BBOX = "651135,6058230,651422,6058548"   # Rodby harbour envelope (meters, EPSG:25832)
resp = requests.get(
    f"{HOST}/collections/{collection_id}/items?bbox={PORT_BBOX}&limit=97")

print("GET MOVING FEATURES BY BBOX:")
print("RESPONSE CODE:", resp.status_code)
resp_json = resp.json()
# print(resp.headers)
print("Response object",json.dumps(resp_json, indent=2))


In [ ]:

from shapely.geometry import box
import geopandas as gpd
data_bbox = resp.json()
features_bbox = resp_json.get("features", [])
geoms = []
ids = []
gdf = json_to_gdf(features_bbox,geoms, ids)
fig = Figure(width="900px", height="400px")
b = fl.Map(location=[54.6, 11.0], zoom_start=6, tiles="OpenStreetMap",width="100%", height="500px")
fig.add_child(b)
fl.GeoJson(
        gdf,
        name="Bbox ships",
        tooltip=fl.GeoJsonTooltip(fields=["id"], aliases=["Ship ID:"]),
        style_function=lambda x: {
                "color": "blue",
                "weight": 1,
                "fillOpacity": 1},
                show=True
).add_to(m)



minx, miny, maxx, maxy = map(float, PORT_BBOX.split(","))

envelope = box(minx, miny, maxx, maxy)

gdf_env = gpd.GeoDataFrame(
    {"name": ["port envelope"]},
    geometry=[envelope],
    crs="EPSG:25832"
).to_crs(4326)
layer = fl.GeoJson(
    gdf_env,
    name="Port envelope",
    tooltip="Rodby",
    style_function=lambda x: {
        "color": "black",
        "weight": 2,
        "fillOpacity": 0.1
    }
).add_to(m)


m


#### 4.GET MOVING FEATURES LIMIT X

In [ ]:
resp = requests.get(f"{HOST}/collections/{collection_id}/items?limit=5")

print("GET SHIPS BY LIMIT:")
print("RESPONSE CODE:", resp.status_code)
resp_json = resp.json()
print("Length of returned objects is:", len(resp_json["features"]))
print("Response object",json.dumps(resp_json, indent=2))
ship_id = resp.json()["features"][4]["id"] # GET ID FOR THE NEXT GET REQUEST -->VELOCITIES

#### 5.GET TEMPORAL GEOM SEQUENCE  

In [ ]:
resp = requests.get(f"{HOST}/collections/{collection_id}/items/{ship_id}/tgsequence")
if resp.status_code != 200:
    print("status isn't 200")

print("GET TEMPORAL GEOM SEQUENCE :")
print("RESPONSE CODE:", resp.status_code)
resp_json = resp.json()
print("Response object",json.dumps(resp_json, indent=2))
geom_id = resp.json()["geometrySequence"][0]["id"] # GET ID FOR THE NEXT GET REQUEST -->VELOCITIES

#### 4.Get velocity progress for one of the ships in the port

In [ ]:
resp = requests.get(
    f"{HOST}/collections/{collection_id}/items/{ship_id}/tgsequence/{geom_id}/velocity"
)
assert resp.status_code == 200
print("GET TEMPORAL GEOM SEQUENCE :")
print("RESPONSE CODE:", resp.status_code)
resp_json = resp.json()
print(f"\nVelocity values for ship {ship_id} limited to 10 instants:")
for point in resp_json .get("values", [])[:10]:
    print(f"  {point['time']}: {point['value']:.2f} m/s")
print("Response object",json.dumps(resp_json, indent=2))


#### 5.Get ships by time interval (subtrajectory parameter between 10:30 and 11:30)

In [ ]:
TIME_INTERVAL = "2024-08-07T10:30:00+00/2024-08-07T11:30:00+00"  

resp = requests.get(
    f"{HOST}/collections/{collection_id}/items",
    params={"subTrajectory": "true", "datetime": TIME_INTERVAL,"limit":"137"}
)
print(resp.url)
print("GET Ships subtrajectory")
print("RESPONSE CODE:", resp.status_code)
resp_json = resp.json()
print("Response object",json.dumps(resp_json, indent=2))


In [ ]:

features = resp_json.get("features", [])

gdf = json_to_gdf(features)
fig = Figure(width="900px", height="400px")
b = fl.Map(location=[54.6, 11.0], zoom_start=6, tiles="cartodbpositron",width="100%", height="500px")
fig.add_child(b)
fl.GeoJson(
    gdf,
    name="Subtrajectories ships 10:30-11:30",
    style_function=lambda x: {"color": "yellow", "weight": 2, "opacity": 1}
).add_to(m)

fl.LayerControl(collapsed=False).add_to(m)
m

In [14]:


collection_id = "ships"
vessel_id ="205196000"
resp = requests.get(f"{HOST}/collections/{collection_id}/items/{vessel_id}")

assert resp.status_code in (200, 404)

print("GET MOVING FEATURES:")
print("RESPONSE CODE:", resp.status_code)
resp_json = resp.json()
# print( len(resp_json["features"]))
# print(resp.headers)
print("Response object",json.dumps(resp_json, indent=2))

GET MOVING FEATURES:
RESPONSE CODE: 404
Response object {
  "code": "404",
  "description": "Feature '205196000' not found in collection 'ships'"
}


In [11]:
collection_id = "ships"
vessel_id ="205196000"

resp = requests.delete(f"{HOST}/collections/{collection_id}/items/{vessel_id}")

print(resp.status_code)


204
